In [ ]:
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus-100", "en-es")
ds["train"] = ds["train"].shuffle(seed=42).select(range(150000))
ds = ds.map(lambda x: {"en": x["translation"]["en"], "es": x["translation"]["es"]}, remove_columns="translation")

In [ ]:
import math
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel as BLPre
from tokenizers.decoders import ByteLevel as BLDec
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import BpeTrainer

MAX_SEQ_LEN=128
TOK_BATCH_SIZE=1000

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = BLPre()
tokenizer.decoder = BLDec()
trainer = BpeTrainer(vocab_size=16_000, special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], initial_alphabet=BLPre.alphabet())

def corpus_iterator():
    for i in range(0, len(ds["train"]), TOK_BATCH_SIZE):
        batch = ds["train"][i:i+TOK_BATCH_SIZE]
        yield batch["en"] + batch["es"]
tokenizer.train_from_iterator(corpus_iterator(), trainer, math.ceil(len(ds["train"])/TOK_BATCH_SIZE))

BOS = tokenizer.token_to_id("[BOS]")
EOS = tokenizer.token_to_id("[EOS]")

tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[("[BOS]", BOS), ("[EOS]", EOS)],
)

In [ ]:
def tokenize(batch):
    en_enc = tokenizer.encode_batch_fast(batch["en"], add_special_tokens=False)
    es_enc = tokenizer.encode_batch_fast(batch["es"], add_special_tokens=True)
    return {"en_ids": [e.ids for e in en_enc], "es_ids": [e.ids for e in es_enc]}

ds = ds.map(tokenize, batched=True)
ds = ds.filter(lambda x: len(x["en_ids"]) <= MAX_SEQ_LEN and len(x["es_ids"]) <= MAX_SEQ_LEN)

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence


PAD = tokenizer.token_to_id("[PAD]")


def collate_fn(batch):
    src = pad_sequence(
        [torch.tensor(sample["en_ids"]) for sample in batch],
        batch_first=True,
        padding_value=PAD
    )
    tgt = pad_sequence(
        [torch.tensor(sample["es_ids"]) for sample in batch],
        batch_first=True,
        padding_value=PAD
    )
    return src, tgt

train_loader = DataLoader(ds["train"], batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(ds["validation"], batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(ds["test"], batch_size=64, shuffle=False, collate_fn=collate_fn)

In [ ]:
from torch import nn

class Attention(nn.Module):
    def __init__(self, model_features: int, query_key_features: int, value_features: int,):
        super().__init__()
        self.W_q = nn.Linear(model_features, query_key_features, bias=False)
        self.W_k = nn.Linear(model_features, query_key_features, bias=False)
        self.W_v = nn.Linear(model_features, value_features, bias=False)
    
    def forward(self, query, key, value, mask=None):
        # IN dimension: (batch, time_step, model_features)
        # OUT dimension: (batch, time_step, value_features)
        q = self.W_q(query)
        k = self.W_k(key)
        scores = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, -1)
        v = self.W_v(value)
        return attn @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, model_features: int, query_key_features: int, value_features: int, num_heads: int):
        super().__init__()
        self.heads = nn.ModuleList([
                Attention(model_features, query_key_features // num_heads, value_features // num_heads) for _ in range(num_heads)
            ])
        self.out_proj = nn.Linear(value_features, model_features, bias=False)
    
    def forward(self, query, key, value, mask=None):
        out = torch.concat([head(query, key, value, mask=mask) for head in self.heads], dim=-1)
        out = self.out_proj(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, model_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(model_features, model_features * 4),
            nn.ReLU(),
            nn.Linear(4 * model_features, model_features)
        )
    
    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, model_features, query_key_features, value_features, num_heads):
        super().__init__()
        self.mha = MultiHeadAttention(model_features, query_key_features, value_features, num_heads)
        self.ln1 = nn.LayerNorm(model_features)
        self.ff = FeedForward(model_features)
        self.ln2 = nn.LayerNorm(model_features)

    def forward(self, x, mask):
        x = self.ln1(x + self.mha(x, x, x, mask=mask))
        x = self.ln2(x + self.ff(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, model_features, query_key_features, value_features, num_heads):
        super().__init__()
        self.mmha = MultiHeadAttention(model_features, query_key_features, value_features, num_heads)
        self.ln1 = nn.LayerNorm(model_features)
        self.mha = MultiHeadAttention(model_features, query_key_features, value_features, num_heads)
        self.ln2 = nn.LayerNorm(model_features)
        self.ff = FeedForward(model_features)
        self.ln3 = nn.LayerNorm(model_features)

    def forward(self, x, encoder_output, self_mask, cross_mask):
        x = self.ln1(x + self.mmha(x, x, x, mask=self_mask))
        x = self.ln2(x + self.mha(x, encoder_output, encoder_output, mask=cross_mask))
        x = self.ln3(x + self.ff(x))
        return x


class EnglishToSpanishLanguageModel(nn.Module):
    def __init__(self, vocab_size, model_features, query_key_features, value_features, num_heads, num_encoders, num_decoders, max_seq_len):
        super().__init__()
        self.enc_token_embeddings = nn.Embedding(vocab_size, model_features, padding_idx=PAD)
        self.enc_pos_embeddings = nn.Embedding(max_seq_len, model_features)
        self.dec_token_embeddings = nn.Embedding(vocab_size, model_features, padding_idx=PAD)
        self.dec_pos_embeddings = nn.Embedding(max_seq_len, model_features)
        self.enc = nn.ModuleList([EncoderBlock(model_features, query_key_features, value_features, num_heads) for _ in range(num_encoders)])
        self.dec = nn.ModuleList([DecoderBlock(model_features, query_key_features, value_features, num_heads) for _ in range(num_decoders)])
        self.linear = nn.Linear(model_features, vocab_size)
    
    def forward(self, src_ids, tgt_ids):
        # In: (B, T_i), (B, T_o)
        enc_tok_emb = self.enc_token_embeddings(src_ids)
        enc_pos_emb = self.enc_pos_embeddings(torch.arange(src_ids.shape[1]))
        src_pad_mask = (src_ids != PAD).unsqueeze(1)
        enc_x = enc_tok_emb + enc_pos_emb
        for e in self.enc:
            enc_x = e(enc_x, mask=src_pad_mask)
        dec_tok_emb = self.dec_token_embeddings(tgt_ids)
        dec_pos_emb = self.dec_pos_embeddings(torch.arange(tgt_ids.shape[1]))
        tgt_pad_mask = (tgt_ids != PAD).unsqueeze(1)
        seq_len = tgt_ids.shape[-1]
        causal_mask = torch.tril(torch.ones((seq_len, seq_len))).bool().unsqueeze(0)
        tgt_mask = tgt_pad_mask & causal_mask
        dec_x = dec_tok_emb + dec_pos_emb
        for d in self.dec:
            dec_x = d(dec_x, enc_x, self_mask=tgt_mask, cross_mask=src_pad_mask)
        return self.linear(dec_x)

model = EnglishToSpanishLanguageModel(tokenizer.get_vocab_size(), 256, 64*8, 64*8, 8, 3, 3, MAX_SEQ_LEN)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

best_loss = float('inf')

for epoch in range(20):
    model.train()

    running_loss = 0.0
    count = 0
    for i, data in enumerate(train_loader):
        src, tgt = data

        optimizer.zero_grad()

        tgt_x = tgt[:, :-1]
        tgt_y = tgt[:, 1:]
        logits = model(src, tgt_x)
        loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_y.reshape(-1))
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        count += 1
        if i % 100 == 0:
            print(f"[epoch: {epoch}] [iter: {i}] Train Loss: {running_loss / count}")
    print(f"[epoch: {epoch}] Train Loss: {running_loss}")
    
    running_loss = 0.0
    model.eval()
    with torch.no_grad():
        for i, data in enumerate(val_loader):
            src, tgt = data
            tgt_x = tgt[:, :-1]
            tgt_y = tgt[:, 1:]
            logits = model(src, tgt_x)
            loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_y.reshape(-1))
            running_loss += loss.item()
        print(f"[epoch: {epoch}] Validation Loss: {running_loss}")
    
        if running_loss < best_loss:
            torch.save(model.state_dict(), f"en_to_es.pt")
            best_loss = running_loss


In [ ]:
model.load_state_dict(torch.load(f"en_to_es.pt", weights_only=True))

running_loss = 0.0
model.eval()
with torch.no_grad():
    for i, data in enumerate(test_loader):
        src, tgt = data
        tgt_x = tgt[:, :-1]
        tgt_y = tgt[:, 1:]
        logits = model(src, tgt_x)
        loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_y.reshape(-1))
        running_loss += loss.item()
    print(f"Test Loss: {running_loss}")

In [ ]:
@torch.no_grad()
def translate(english: str, max_len=MAX_SEQ_LEN) -> str:
    model.eval()

    en_ids = torch.tensor([tokenizer.encode(english).ids])
    es_ids = torch.tensor([[BOS]])

    for _ in range(max_len):
        logits = model(en_ids, es_ids)
        next_tok = logits[0, -1].argmax()
        es_ids = torch.cat((es_ids, next_tok.view(1, 1)), dim=-1)
        if next_tok.item() == EOS:
            break

    return tokenizer.decode(es_ids[0].tolist())

In [ ]:
translate("What country do you want to travel to?")